In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    root_mean_squared_log_error,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

In [ ]:
train = pd.read_csv("../data/raw/train.csv")
train = train.sort_values("date", ascending=True).reset_index(drop=True)

In [10]:
train["date"] = pd.to_datetime(train["date"])
train["onpromotion"] = train["onpromotion"].astype("category")
train["store_nbr"] = train["store_nbr"].astype("category")
train["family"] = train["family"].astype("category")
train["weekday"] = train["date"].dt.day_of_week
train["month"] = train["date"].dt.month
train["week"] = train["date"].dt.isocalendar().week
train["year"] = train["date"].dt.year

In [45]:
import sys

print(sys.executable)

/Users/silviadeanituaaspiazu/Library/Caches/pypoetry/virtualenvs/storesales-timeseriesforecasting-5H9Z9Y-8-py3.15/bin/python


In [33]:
train["sales_lag15"] = train.groupby(["family", "store_nbr"])["sales"].shift(15)
train["sales_lag365"] = train.groupby(["family", "store_nbr"])["sales"].shift(365)
train["sales_lag30"] = train.groupby(["family", "store_nbr"])["sales"].shift(30)

train["sales_lag15_log"] = np.log1p(train["sales_lag15"])
train["sales_lag365_log"] = np.log1p(train["sales_lag365"])
train["sales_lag30_log"] = np.log1p(train["sales_lag30"])


df = pd.get_dummies(train, columns=["store_nbr", "family"])
dummy_cols = [
    col for col in df.columns if col.startswith("family") or col.startswith("store_nbr")
]

features = [
    "onpromotion",
    "store_nbr",
    "family",
    "sales_lag15_log",
    "sales_lag365_log",
    "sales_lag30_log",
] + dummy_cols
y = "sales"

horizon = 15
horizon_days = pd.Timedelta(days=15)

model = Pipeline(
    steps=[
        (
            "model",
            TransformedTargetRegressor(
                regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1
            ),
        )
    ]
)

cutoffs = ["2017-07-15", "2017-04-15", "2017-01-15", "2016-09-15"]
cutoffs = pd.to_datetime(cutoffs)


results = []
for cutoff in cutoffs:
    train_df = train[train["date"] < cutoff]
    test_df = train[
        (train["date"] >= cutoff) and (train["date"] <= (cutoff + horizon_days))
    ]

    X_train, X_test = train_df[features], test_df[features]
    y_train, y_test = train_df[y], test_df[y]

    model.fit(X_train, y_train)
    y_pred = np.clip(model.predict(X_test), a_min=0)

    X_train["cutoff"] = cutoff
    X_train["y_pred"] = y_pred

    mae = mean_absolute_error(y_true=y_test, y_pred=y_pred)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_pred))

    print(f"Cutoff: {cutoff}, MAE: {mae:.2f}, RMSE:{rmse:.2f}")
    results.append(X_train)
df_results = pd.concat(results)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.item(), a.any() or a.all().

In [ ]:
train["date"].max()

Timestamp('2017-08-15 00:00:00')

In [37]:
cutoffs

DatetimeIndex(['2017-07-15', '2017-04-15', '2017-01-15', '2016-09-15'],
              dtype='datetime64[us]', freq=None)

In [ ]:
horizon_f = cu + horizon_days
hroizon_f

TypeError: can only concatenate str (not "Timedelta") to str

In [38]:
train.dtypes

id                           int64
date                datetime64[us]
store_nbr                 category
family                    category
sales                      float64
onpromotion               category
weekday                      int32
month                        int32
week                        UInt32
year                         int32
sales_lag15                float64
sales_lag365               float64
sales_lag30                float64
sales_lag15_log            float64
sales_lag365_log           float64
sales_lag30_log            float64
dtype: object

In [36]:
cutoffs = pd.to_datetime(cutoffs)

test_df = train[
    (train["date"] >= cutoff[0]) & (train["date"] <= (cutoff[0] + horizon_days))
]

TypeError: Invalid comparison between dtype=datetime64[us] and str

In [31]:
test_df = train[(train["date"] >= cutoff)]
test_df

,id,date,store_nbr,family,sales,onpromotion,weekday,month,week,year,sales_lag15,sales_lag365,sales_lag30,sales_lag15_log,sales_lag365_log,sales_lag30_log
2943864,2945047,2017-07-15,41,POULTRY,212.475,0,5,7,28,2017,201.372,160.422000,164.194,5.310108,5.084022,5.107121
2943865,2945055,2017-07-15,42,BEVERAGES,4181.000,49,5,7,28,2017,2617.000,1744.000000,1989.000,7.870166,7.464510,7.595890
2943866,2945054,2017-07-15,42,BEAUTY,4.000,1,5,7,28,2017,0.000,4.000000,3.000,0.000000,1.609438,1.386294
2943867,2945053,2017-07-15,42,BABY CARE,0.000,0,5,7,28,2017,0.000,0.000000,0.000,0.000000,0.000000,0.000000
2943868,2945052,2017-07-15,42,AUTOMOTIVE,8.000,0,5,7,28,2017,9.000,1.000000,4.000,2.302585,0.693147,1.609438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000883,2999693,2017-08-15,25,PET SUPPLIES,3.000,0,1,8,33,2017,1.000,10.000000,6.000,0.693147,2.397895,1.945910
3000884,2999692,2017-08-15,25,PERSONAL CARE,93.000,8,1,8,33,2017,98.000,98.000000,120.000,4.595120,4.595120,4.795791
3000885,2999691,2017-08-15,25,MEATS,180.107,0,1,8,33,2017,183.395,112.264000,122.519,5.217080,4.729721,4.816395
3000886,2999702,2017-08-15,26,BEAUTY,0.000,0,1,8,33,2017,0.000,0.000000,2.000,0.000000,0.000000,1.098612
